# Workshop Cleanup

## Overview

The labs save creation records so you can clean up the resources belonging to **one workshop run**. This notebook first shows a local plan, then optionally verifies and deletes those recorded resources.

**What you'll do:**

- Find the saved creation manifest for your run
- Review the exact resources selected for cleanup
- Stop background work and execute the reviewed plan when ready
- Read completed, retained, or unfinished steps

With both switches false, planning is local. **`VERIFY_OWNERSHIP=True`** performs read-only AWS checks; **`EXECUTE_CLEANUP=True`** verifies and deletes the selected resources. The plan covers recorded exercise resources, not every resource in the account.

Keep shared workshop infrastructure until the lessons are finished. Stop traffic and active CodeBuild jobs before removing their dependencies.


In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'workshop_utils').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "03-developer-journey"))
from utils.cleanup import CleanupError, cleanup, manifest_from_records, plan_cleanup

VERIFY_OWNERSHIP = False
EXECUTE_CLEANUP = False
MANIFEST_PATH = ROOT / '.cleanup-manifest.json'
JOURNEY_STATE_PATH = ROOT / '03-developer-journey/.journey-run.json'
# Select actual files below after reviewing the inventory; nothing is auto-authorized.
SELECTED_RECORDS = []
manifest = None
print({'verify_ownership': VERIFY_OWNERSHIP, 'execute_cleanup': EXECUTE_CLEANUP})

## Step 1: Inspect Saved Resources

Each deployment or managed experiment writes a creation record with its run ID and exact resource IDs. The next cell inventories local files; it does not query or delete AWS resources.

Look for a **`cleanup_manifest`**, such as `.cleanup-deployment-<agent>.json` or `.cleanup-managed-<run>.json`. These contain the creation records the planner needs. Raw `.managed-*.json` ledgers and `.lifecycle-runs/` evaluation results help you review the experiment, but evaluation results alone are not evidence that this run created a resource.

Match the run you intend to finish with its saved manifest before selecting anything for deletion.


In [ ]:
journey_state = json.loads(JOURNEY_STATE_PATH.read_text()) if JOURNEY_STATE_PATH.is_file() else None
available = []
for directory in (ROOT, ROOT / '03-developer-journey'):
    for pattern in ('.journey-run.json', '.deployment-*.json', '.gateway-*.json',
                    '.managed-*.json', '.cleanup-*.json'):
        available.extend(directory.glob(pattern))
for name in ('gate.json', 'ab_gate.json', 'promotion.json', 'rollback.json',
             '.cleanup-*.json', '.managed-*.json'):
    available.extend((ROOT / '.lifecycle-runs').glob('*/' + name))
if MANIFEST_PATH.is_file():
    available.append(MANIFEST_PATH)
available = sorted({path for path in available if path.is_file()})

for path in available:
    value = json.loads(path.read_text())
    is_cleanup_manifest = (
        value.get('schema_version') == 1
        and all(key in value for key in ('run_id', 'account_id', 'region', 'resources'))
    )
    print({'file': str(path.relative_to(ROOT)), 'run_id': value.get('run_id'),
           'record_type': 'cleanup_manifest' if is_cleanup_manifest else 'supporting_record',
           'resource_count': len(value.get('resources', [])),
           'action_needed': value.get('action_needed', []),
           'runtime_arn': value.get('agentRuntimeArn'), 'gateway_id': value.get('gateway_id')})
if not available:
    print('No local deployment/lifecycle/cleanup records found. No cleanup plan or AWS action was inferred.')

# Prefer actual scope from persisted state. Fill account from your recorded deployment,
# or supply the account you expect; verification will compare it with STS.
RUN_ID = journey_state.get('run_id', '') if journey_state else os.environ.get('WORKSHOP_RUN_ID', '')
REGION = journey_state.get('region', '') if journey_state else os.environ.get('AWS_REGION', os.environ.get('AWS_DEFAULT_REGION', ''))
ACCOUNT_ID = os.environ.get('WORKSHOP_ACCOUNT_ID', '')
print({'recorded_run_id': RUN_ID, 'expected_account_id': ACCOUNT_ID, 'recorded_region': REGION})

## Step 2: Select the Creation Manifest

Set `MANIFEST_PATH` to an inventoried `cleanup_manifest`, then rerun the setup, inventory, and planning cells in order. For example, select `ROOT / "03-developer-journey/.cleanup-managed-<run>.json"` using your actual run ID.

Alternatively, populate `SELECTED_RECORDS` with exact creation records/cleanup manifests from the inventory and set the expected account, Region, and run. An existing `MANIFEST_PATH` takes precedence; leave it pointing to a nonexistent file when using that alternative.

Read the resulting plan before execution:

| Plan detail | What to check |
|---|---|
| **Run/account/Region** | The intended workshop run |
| **Resource IDs** | The deployments and experiments you finished |
| **Artifact versions** | Exact uploaded objects recorded for that run |
| **Retained resources** | Shared or CloudFormation-owned infrastructure |
| **Action needed** | Missing creation evidence or an unresolved record |

Use the saved records as they were produced. A name match alone does not establish that a resource belongs to this run, and incomplete evidence leaves cleanup blocked until resolved.


In [ ]:
if MANIFEST_PATH.is_file():
    manifest = json.loads(MANIFEST_PATH.read_text())
    # plan_cleanup validates scope/identities; no AWS calls.
    plan = plan_cleanup(manifest)
elif SELECTED_RECORDS:
    if not all((RUN_ID, ACCOUNT_ID, REGION)):
        raise ValueError('Supply the recorded run, account and Region before importing creation records')
    manifest = manifest_from_records(SELECTED_RECORDS, run_id=RUN_ID, account_id=ACCOUNT_ID, region=REGION)
    plan = plan_cleanup(manifest)
else:
    plan = None
    print('No explicit manifest selected; nothing is authorized for cleanup.')

if plan is not None:
    print(json.dumps(plan, indent=2))
    if plan['action_needed']:
        print('BLOCKED until the listed creation/ownership evidence is supplied.')
    elif not plan['steps']:
        print('This manifest has no runtime-created deletion steps; this is not an account-wide cleanliness check.')

## Step 3: Verify and Execute the Plan

Leave both switches false to review locally. Enable **only `VERIFY_OWNERSHIP`** for read-only checks. When ready to delete the reviewed resources, enable **`EXECUTE_CLEANUP`**; execution repeats ownership verification before making changes.

The helper stops A/B tests and online evaluation first, then removes their configurations, Gateway targets before Gateways, bundles before Runtimes, and exact recorded artifacts. This order avoids leaving a dependency active after its target is removed.

Execution also shuts down this kernel's workshop telemetry exporters. Stop other active notebook kernels/exporters before later shared-resource teardown. Closing a notebook tab alone does not stop its kernel.

Read the returned report. A timeout or failed deletion can leave work unfinished; it does not mean the resource disappeared. The same manifest can be rerun to check actual state and resume remaining steps.


In [ ]:
report = None
if VERIFY_OWNERSHIP or EXECUTE_CLEANUP:
    if manifest is None:
        raise ValueError('Select a real creation manifest before enabling AWS checks')
    if not plan['steps']:
        raise ValueError('No runtime-created resources selected; no AWS cleanup is needed')
    if EXECUTE_CLEANUP:
        from workshop_utils.observability import shutdown_observability
        # Stop this kernel's ADOT trace/log/metric exporters before cloud cleanup.
        # Any shutdown failure stops this cell before AWS clients or mutations.
        shutdown_observability()
        print('Workshop observability shutdown completed for this kernel only.')
    import boto3
    from botocore.config import Config
    session = boto3.Session(region_name=manifest['region'])
    config = Config(connect_timeout=3, read_timeout=15, retries={'total_max_attempts': 2})
    clients = {service: session.client(service, config=config) for service in (
        'sts', 'cloudformation', 'bedrock-agentcore-control', 'bedrock-agentcore', 'bedrock', 's3')}
    failure = None
    try:
        report = cleanup(manifest, clients, execute=EXECUTE_CLEANUP, verify=VERIFY_OWNERSHIP,
                         timeout_s=300, poll_interval_s=2)
    except CleanupError as exc:
        report, failure = exc.report, exc
    # Save actual results even after partial failure; never replace them with success.
    report_dir = ROOT / '.cleanup-reports'
    report_dir.mkdir(exist_ok=True)
    # Use the manifest digest to avoid interpreting run IDs as filesystem paths.
    report_path = report_dir / (report['manifest_sha256'] + '.json')
    report_path.write_text(json.dumps(report, indent=2) + '\n')
    print(json.dumps(report, indent=2))
    print(f'Actual cleanup report: {report_path}')
    if failure is not None:
        raise failure
else:
    print('DRY RUN ONLY: no AWS clients, ownership assertions, stop/disable calls or deletions performed.')

## Step 4: Read the Cleanup Report

| Status | Meaning |
|---|---|
| `verified_plan` | Read-only checks passed; nothing was deleted |
| `blocked` | No mutations occurred; inspect the listed issue |
| `partial_failure` | Some steps finished; inspect failed and unattempted steps |
| `already_absent` | The exact resource was reported not found |
| `complete` | The selected steps finished or were already absent |

Retained CloudFormation/shared resources remain outside this plan. Keep the report and creation records until cleanup is complete. A complete plan does not mean the whole account is empty.

### Finish the Workshop

Stop all active jobs, traffic, and notebook exporters before shared infrastructure teardown. In each kernel that initialized workshop telemetry, run its shutdown cell or shut down the kernel. The current notebook can stop exporters only in its own process.

### Self-Paced Stack Teardown

After recorded resources are cleaned up, save any needed results outside Code Editor. Return to the **original CloudShell or local terminal outside Code Editor** and run from the repository root:

```bash
make -C 03-developer-journey delete-all RUNTIME_RESOURCES_CLEANED=1
```

This removes gate → Code Editor → Langfuse → core → Cognito. Run it outside Code Editor because that instance is itself deleted. The template-upload bucket and account-level Transaction Search settings are retained.
